# Imports

In [ ]:
from fastai.vision.all import *
# from fastai.distributed import * #For distributing across multi GPU
# from sklearn.metrics import roc_curve, auc 
# from fastai.metrics import * 
from sklearn.model_selection import RepeatedKFold, GroupKFold, GroupShuffleSplit, \
                                    StratifiedGroupKFold, LeaveOneGroupOut, LeavePGroupsOut
from sklearn.utils import resample
import sklearn.metrics as skm
from pathlib import Path
import numpy as np
from numpy import random
import shutil
import glob
import os
import pandas as pd
from torch import cuda
import gc
import shutil
import time
from tqdm import tqdm
state = 36
scratch = os.getenv('SLURM_SCRATCH')
print(scratch)


# Import data

In [ ]:
df = pd.read_csv('/path/to/balanced_sk_lu_lv_cr_df_v1_tiles.tsv',
                 sep = '\t')  ## user is responseible to update this tsv file name accordingly
n = 1000  ## user needs to update this number accordingly
df.loc[:,'tissue_anno'] = df.tissue + df.anno
df_ds = resample(df,n_samples=n,
             random_state=state, 
             replace=False,
             stratify = df.tissue_anno)
df_ds = df_ds.reset_index(drop=True)
print(df_ds.shape)
df_ds.loc[:,'scratch_fn'] = scratch + '/' + df_ds.fn.str.split('/').str[-1]
for i,fn in enumerate(tqdm(df_ds.fn.values)):
    scratch_fn = df_ds.loc[i,'scratch_fn']
    try:
        shutil.copyfile(fn,scratch_fn)
    except:
        print(scratch_fn,'Copy Failed')
print(df_ds.scratch_fn.isna().sum(),'missing')
df_ds.groupby(['tissue','anno'])['fn'].count()

# Show 36 training examples

In [ ]:
state = 36
splitter = TrainTestSplitter(test_size=0.1, random_state=state, stratify=df_ds.tissue_anno.values,
                    train_size=None, shuffle=True)
batch_size = 200 
tissue =DataBlock(blocks=(ImageBlock, CategoryBlock),
                  get_x=ColReader('scratch_fn'),
                  splitter=splitter,
                  get_y=  ColReader('tissue_anno'),
                  item_tfms=Resize(460), #Presize
                  batch_tfms=aug_transforms(size=224,
                                            max_rotate=45, # size=224,
                                            min_scale=1,
                                            max_zoom=0,
                                            flip_vert=True,
                                           )
                             ) 
dls = tissue.dataloaders(df_ds, bs = batch_size)
dls.show_batch(max_n=36)